In [ ]:
# install required packages (only need to run once)
!pip install torch torchvision matplotlib tqdm

# Python imports
import os
from glob import glob
from PIL import Image
from tqdm.auto import tqdm

# PyTorch imports
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader

# TorchVision transforms & utilities
import torchvision.transforms as T
import torchvision.utils as vutils

# Matplotlib for visualization
import matplotlib.pyplot as plt


In [ ]:
class MapsDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        root_dir: path to folder with side-by-side images
        Each image is expected to be [input | target] format.
        """
        self.files = sorted(glob(os.path.join(root_dir, "*.*")))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        w, h = img.size
        w2 = w // 2

        img_A = img.crop((0, 0, w2, h))     # left half: input
        img_B = img.crop((w2, 0, w, h))     # right half: target

        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)

        return img_A, img_B


In [ ]:
root_dir = "/kaggle/input/pix2pix-maps/train"  # or wherever your data lives

transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds = MapsDataset(root_dir=root_dir, transform=transform)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)


In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, features=64):
        super().__init__()
        def down(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
            if norm: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)
        def up(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.down1 = down(in_ch, features, norm=False)
        self.down2 = down(features,  features*2)
        self.down3 = down(features*2,features*4)
        self.down4 = down(features*4,features*8)
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features*8, features*8, 4, 2, 1),
            nn.ReLU(inplace=True)
        )
        # Decoder
        self.up1 = up(features*8, features*8)
        self.up2 = up(features*16,features*4)  # skip concat doubles channels
        self.up3 = up(features*8, features*2)
        self.up4 = up(features*4, features)
        self.final = nn.ConvTranspose2d(features*2, out_ch, 4, 2, 1)
        self.tanh  = nn.Tanh()

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        bn = self.bottleneck(d4)
        u1 = self.up1(bn);   u1 = torch.cat([u1, d4], dim=1)
        u2 = self.up2(u1);   u2 = torch.cat([u2, d3], dim=1)
        u3 = self.up3(u2);   u3 = torch.cat([u3, d2], dim=1)
        u4 = self.up4(u3);   u4 = torch.cat([u4, d1], dim=1)
        out = self.final(u4)
        return self.tanh(out)


In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch=6, features=64):
        super().__init__()
        def block(in_c, out_c, stride):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 4, stride, 1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.LeakyReLU(0.2, inplace=True)
            )
        self.model = nn.Sequential(
            block(in_ch, features, stride=2),
            block(features, features*2, 2),
            block(features*2,features*4,2),
            block(features*4,features*8,1),
            nn.Conv2d(features*8, 1, 4, 1, 1)  # output single-channel patch map
        )

    def forward(self, a, b):
        x = torch.cat([a, b], dim=1)
        return self.model(x)


In [ ]:
def show_tensor_images(image_tensor, num_images=25, size=(1, 28, 28)):
    '''
    Function for visualizing images: Given a tensor of images, number of images, and
    size per image, plots and prints the images in an uniform grid.
    '''
    image_shifted = (image_tensor + 1) / 2
    image_shifted = image_tensor
    image_unflat = image_shifted.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=4)
    plt.imshow(image_grid.permute(1, 2, 0).squeeze())
    plt.show()

In [ ]:
def train(epochs):
    for ep in range(1, epochs+1):
        loop = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}")
        for real_A, real_B in loop:
            real_A, real_B = real_A.to(device), real_B.to(device)
            bs = real_A.size(0)

            #— 1) Train Discriminator —#
            fake_B = G(real_A).detach()
            D_real = D(real_A, real_B)
            D_fake = D(real_A, fake_B)
            real_lbl = torch.ones_like(D_real)
            fake_lbl = torch.zeros_like(D_fake)
            loss_D = 0.5*(criterion_GAN(D_real, real_lbl) + criterion_GAN(D_fake, fake_lbl))
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            #— 2) Train Generator —#
            fake_B = G(real_A)
            D_fake2 = D(real_A, fake_B)
            loss_G_GAN = criterion_GAN(D_fake2, real_lbl)
            loss_G_L1  = criterion_L1(fake_B, real_B)*100
            loss_G = loss_G_GAN + loss_G_L1
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            loop.set_postfix(loss_D=loss_D.item(), loss_G=loss_G.item())
        # save samples each epoch
        with torch.no_grad():
            sample_and_save(ep, real_A, fake_B, real_B)

def sample_and_save(ep, A, fakeB, B, folder="samples"):
    os.makedirs(folder, exist_ok=True)
    imgs = torch.cat([A, fakeB, B], dim=0)*0.5+0.5
    grid = vutils.make_grid(imgs, nrow=A.size(0))
    plt.figure(figsize=(8,4))
    plt.imshow(grid.permute(1,2,0).cpu())
    plt.axis('off')
    plt.title(f"Epoch {ep}")
    plt.savefig(f"{folder}/epoch_{ep}.png")
    plt.close()


In [ ]:
# --- New Debug Cell: check one forward pass through G ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
G = UNetGenerator().to(device)

real_A, real_B = next(iter(train_loader))
print("real_A shape:", real_A.shape)

real_A = real_A.to(device)
try:
    fake_B = G(real_A)
    print("fake_B shape:", fake_B.shape)
except Exception as e:
    print("🚨 Error in G(real_A):", e)
    raise


In [ ]:
# --- Cell 8 (Training & Save) ---
D = PatchDiscriminator().to(device)
# Load saved weights if available
G_path = "/kaggle/input/image-to-image-translation/pytorch/default/1/generator.pth"
D_path = "/kaggle/input/image-to-image-translation/pytorch/default/1/discriminator.pth"

if os.path.exists(G_path):
    G.load_state_dict(torch.load(G_path, map_location=device))
    print("✅ Loaded Generator weights.")

if os.path.exists(D_path):
    D.load_state_dict(torch.load(D_path, map_location=device))
    print("✅ Loaded Discriminator weights.")

# Optimizers and losses
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
criterion_GAN = nn.BCEWithLogitsLoss()
criterion_L1  = nn.L1Loss()

# Train (continue training)
def train(epochs):
    cur_step = 0
    for ep in range(1, epochs+1):
        loop = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}")
        for real_A, real_B in loop:
            real_A, real_B = real_A.to(device), real_B.to(device)

            # Discriminator step
            fake_B = G(real_A).detach()
            D_real = D(real_A, real_B)
            D_fake = D(real_A, fake_B)
            real_lbl = torch.ones_like(D_real)
            fake_lbl = torch.zeros_like(D_fake)
            loss_D = 0.5*(criterion_GAN(D_real, real_lbl) + criterion_GAN(D_fake, fake_lbl))
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            # Generator step
            fake_B2 = G(real_A)
            D_fake2 = D(real_A, fake_B2)
            loss_G = criterion_GAN(D_fake2, real_lbl) + criterion_L1(fake_B2, real_B)*100
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            loop.set_postfix(loss_D=loss_D.item(), loss_G=loss_G.item())
            cur_step += 1
        # save sample per epoch
        with torch.no_grad():
            sample_and_save(ep, real_A, fake_B2, real_B)
        
    # final save
    torch.save(G.state_dict(), "generator.pth")
    torch.save(D.state_dict(), "discriminator.pth")
    print("✅ Training complete and models saved.")

1# run!
train(epochs=100)
